# 02 — Building success rate from scratch

"Success rate" is the foundational advanced-stats metric: instead of judging a play by yards gained alone, judge it against the yards it *needed* to gain. The standard (Bill Connelly) definition:

- 1st down: gain >= 50% of yards-to-go
- 2nd down: gain >= 70% of yards-to-go
- 3rd/4th down: gain >= 100% of yards-to-go (i.e. convert)
- Touchdowns always succeed; turnovers always fail

This notebook derives it by hand on a few example plays, then checks that against `cfb_analytics.metrics`, then applies it to a full season.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

examples = pd.DataFrame([
    {"down": 1, "distance": 10, "yardsGained": 6, "playType": "Rush"},
    {"down": 2, "distance": 7, "yardsGained": 4, "playType": "Pass Reception"},
    {"down": 3, "distance": 3, "yardsGained": 3, "playType": "Rush"},
    {"down": 3, "distance": 8, "yardsGained": 8, "playType": "Rushing Touchdown"},
    {"down": 2, "distance": 5, "yardsGained": 25, "playType": "Interception"},
])
examples

,down,distance,yardsGained,playType
0,1,10,6,Rush
1,2,7,4,Pass Reception
2,3,3,3,Rush
3,3,8,8,Rushing Touchdown
4,2,5,25,Interception


In [2]:
# By hand: compute the threshold for each row's down, then compare.
def threshold(row):
    if row["down"] == 1:
        return 0.5 * row["distance"]
    if row["down"] == 2:
        return 0.7 * row["distance"]
    return row["distance"]

examples["threshold"] = examples.apply(threshold, axis=1)
examples["met_threshold"] = examples["yardsGained"] >= examples["threshold"]
examples

,down,distance,yardsGained,playType,threshold,met_threshold
0,1,10,6,Rush,5.0,True
1,2,7,4,Pass Reception,4.9,False
2,3,3,3,Rush,3.0,True
3,3,8,8,Rushing Touchdown,8.0,True
4,2,5,25,Interception,3.5,True


Row 4 (the touchdown) already meets its threshold by yardage, so it's not a great test of the touchdown rule. Row 5 (the interception) *does* meet its threshold by yardage (25 >= 3.5) but should still be a failure — turnovers override the yardage math. This is exactly why `metrics.py` special-cases touchdowns and turnovers instead of relying on `yardsGained` alone.

In [3]:
from cfb_analytics.metrics import add_all_metrics

checked = add_all_metrics(examples[["down", "distance", "yardsGained", "playType"]])
checked[["down", "distance", "yardsGained", "playType", "success"]]

,down,distance,yardsGained,playType,success
0,1,10,6,Rush,1.0
1,2,7,4,Pass Reception,0.0
2,3,3,3,Rush,1.0
3,3,8,8,Rushing Touchdown,1.0
4,2,5,25,Interception,0.0


Matches the hand calculation for rows 1-3, and correctly flips row 5 (interception) to a failure despite the big gain. Now apply it to a full season and look at the national leaderboard.

In [4]:
from cfb_analytics.client import get_season_plays
from cfb_analytics.aggregate import team_season_splits, national_averages

plays = get_season_plays(2024)
splits = team_season_splits(plays)
offense = splits["offense"]
offense[offense["plays"] >= 200].sort_values("success_rate", ascending=False).head(15)

,team,plays,success_rate,stuff_rate,havoc_rate,ppa_per_play
99,Indiana,785,0.527389,0.091720,0.121019,0.317474
219,South Dakota,687,0.522562,0.082969,0.104803,0.368208
150,Montana State,832,0.521635,0.085337,0.099760,0.372246
67,Eastern Washington,703,0.520626,0.056899,0.079659,0.315750
137,Miami,853,0.516999,0.039859,0.078546,0.378271
187,Penn State,852,0.514085,0.076291,0.104460,0.288155
14,Army,706,0.512748,0.067989,0.083569,0.308195
168,North Dakota State,851,0.512338,0.096357,0.117509,0.303976
185,Oregon,881,0.507378,0.066969,0.088536,0.290994
292,William & Mary,627,0.499203,0.082935,0.125997,0.231342


In [5]:
national_averages(offense)

success_rate    0.401045
stuff_rate      0.093531
havoc_rate      0.140727
ppa_per_play    0.070168
dtype: float64

Next: `scripts/build_dataset.py` runs this same pipeline for a full season and writes the processed tables the dashboard reads; `scripts/train_baseline_model.py` uses these situational features (down, distance, field position, score) to predict individual play success — the first real ML model in the project.